In [2]:
# --- ÉTAPE 0 : IMPORTATION DES BIBLIOTHÈQUES ---
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import joblib

# Modules Scikit-Learn
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.ensemble import RandomForestClassifier
from sklearn.tree import DecisionTreeClassifier  # <--- AJOUTÉ (Modèle DT)
from sklearn.impute import SimpleImputer         # <--- AJOUTÉ (Valeurs manquantes)
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score

# Module Imbalanced-Learn
from imblearn.over_sampling import SMOTE

pd.set_option('display.max_columns', None)
import warnings
warnings.filterwarnings('ignore')

print("✅ Bibliothèques chargées (DT et Imputer inclus) !")

✅ Bibliothèques chargées (DT et Imputer inclus) !


In [4]:
# --- ÉTAPE 1 & 2 : CHARGEMENT ET SIMULATION (SCENARIO FACILE) ---

print("--- DÉBUT DU SCÉNARIO COMPARATIF (16% DÉFAUTS) ---")

# 1. Chargement du fichier
df_16 = pd.read_csv('manufacturing_defect_dataset.csv')

# 2. Correction de la Logique (Indispensable)
# Dans ce dataset : 0 = Défaut, 1 = Normal
# On inverse pour que 1 = Défaut (Standard universel)
df_16['DefectStatus'] = df_16['DefectStatus'].replace({0: 1, 1: 0})

# 3. Vérification de la Distribution
# Ici, on ne fait PAS de 'sample'. On garde tout le monde.
count_defauts = df_16['DefectStatus'].sum()
total_lignes = len(df_16)
pourcentage = (count_defauts / total_lignes) * 100

print(f"Total lignes : {total_lignes}")
print(f"Nombre de défauts (Classe 1) : {count_defauts}")
print(f"Pourcentage de défauts : {pourcentage:.2f}%")
print("👉 Note : Avec 16%, le problème est considéré comme 'Facile' ou 'Modéré'.")

--- DÉBUT DU SCÉNARIO COMPARATIF (16% DÉFAUTS) ---
Total lignes : 3240
Nombre de défauts (Classe 1) : 517
Pourcentage de défauts : 15.96%
👉 Note : Avec 16%, le problème est considéré comme 'Facile' ou 'Modéré'.


In [5]:
# --- ÉTAPE 3 : PRÉTRAITEMENT (IMPUTER, SPLIT & SMOTE) ---

# 1. Séparation des Features (X) et de la Cible (y)
X_16 = df_16.drop('DefectStatus', axis=1)
y_16 = df_16['DefectStatus']

# --- AJOUT : GESTION DES VALEURS MANQUANTES ---
# On remplace les trous par la médiane (plus robuste)
print("🛠️ Nettoyage des valeurs manquantes...")
imputer = SimpleImputer(strategy='median')
X_16 = pd.DataFrame(imputer.fit_transform(X_16), columns=X_16.columns)
# -----------------------------------------------

# 2. Division Train / Test
X_train_16, X_test_16, y_train_16, y_test_16 = train_test_split(
    X_16, y_16, test_size=0.3, stratify=y_16, random_state=42
)

# 3. SMOTE (Uniquement sur le Train)
smote = SMOTE(random_state=42)
X_train_smote_16, y_train_smote_16 = smote.fit_resample(X_train_16, y_train_16)

print(f"Taille Train Original : {y_train_16.shape[0]}")
print(f"Taille Train Après SMOTE : {y_train_smote_16.shape[0]}")

🛠️ Nettoyage des valeurs manquantes...
Taille Train Original : 2268
Taille Train Après SMOTE : 3812


In [6]:
# --- ÉTAPE 4 & 5 : ENTRAÎNEMENT ET ÉVALUATION (DT & RF) ---

# === 1. DECISION TREE (DT) ===
print("\n--- 1. ENTRAÎNEMENT DECISION TREE (DT) ---")
dt_model = DecisionTreeClassifier(random_state=42)
dt_model.fit(X_train_smote_16, y_train_smote_16)

# Test rapide du DT
y_pred_dt = dt_model.predict(X_test_16)
print("RÉSULTATS : DECISION TREE")
print(classification_report(y_test_16, y_pred_dt))
print("-" * 40)

# === 2. RANDOM FOREST (RF) ===
print("\n--- 2. ENTRAÎNEMENT RANDOM FOREST (RF) ---")
rf_base_16 = RandomForestClassifier(n_estimators=100, random_state=42)
rf_base_16.fit(X_train_smote_16, y_train_smote_16)

# Test du RF
y_pred_base_16 = rf_base_16.predict(X_test_16)

print("RÉSULTATS : RANDOM FOREST (BASE)")
print(classification_report(y_test_16, y_pred_base_16))

# Matrice de confusion (RF)
conf_matrix = confusion_matrix(y_test_16, y_pred_base_16)
print(f"Vrais Défauts trouvés (RF) : {conf_matrix[1,1]} sur {sum(y_test_16)}")


--- 1. ENTRAÎNEMENT DECISION TREE (DT) ---
RÉSULTATS : DECISION TREE
              precision    recall  f1-score   support

           0       0.96      0.88      0.92       817
           1       0.55      0.79      0.65       155

    accuracy                           0.86       972
   macro avg       0.75      0.83      0.78       972
weighted avg       0.89      0.86      0.87       972

----------------------------------------

--- 2. ENTRAÎNEMENT RANDOM FOREST (RF) ---
RÉSULTATS : RANDOM FOREST (BASE)
              precision    recall  f1-score   support

           0       0.96      0.99      0.97       817
           1       0.93      0.78      0.85       155

    accuracy                           0.96       972
   macro avg       0.95      0.88      0.91       972
weighted avg       0.96      0.96      0.95       972

Vrais Défauts trouvés (RF) : 121 sur 155


In [5]:
# --- ÉTAPE 6 : OPTIMISATION AVANCÉE (GRIDSEARCH) ---

print("⏳ Lancement de l'optimisation (GridSearch)... Patientez...")

# 1. Définir l'espace de recherche (Paramètres à tester)
param_grid = {
    'n_estimators': [100, 200],
    'max_depth': [10, None],        # Profondeur de l'arbre
    'min_samples_leaf': [1, 2],     # Minimum d'exemples par feuille
    'class_weight': ['balanced', None]
}

# 2. Lancer la recherche du meilleur modèle
grid_search_16 = GridSearchCV(
    RandomForestClassifier(random_state=42),
    param_grid,
    scoring='recall', # On veut maximiser la détection des défauts
    cv=3,             # Validation croisée (3 fois)
    n_jobs=-1         # Utiliser tous les processeurs
)

grid_search_16.fit(X_train_smote_16, y_train_smote_16)

# 3. Récupérer le champion
best_rf_16 = grid_search_16.best_estimator_

print(f"✅ Meilleurs paramètres trouvés : {grid_search_16.best_params_}")

# 4. Test Final avec le modèle optimisé
y_pred_opt_16 = best_rf_16.predict(X_test_16)

print("\n--- RÉSULTATS : MODÈLE OPTIMISÉ (16%) ---")
print(classification_report(y_test_16, y_pred_opt_16))

⏳ Lancement de l'optimisation (GridSearch)... Patientez...
✅ Meilleurs paramètres trouvés : {'class_weight': 'balanced', 'max_depth': None, 'min_samples_leaf': 1, 'n_estimators': 200}

--- RÉSULTATS : MODÈLE OPTIMISÉ (16%) ---
              precision    recall  f1-score   support

           0       0.96      0.98      0.97       817
           1       0.91      0.80      0.85       155

    accuracy                           0.95       972
   macro avg       0.93      0.89      0.91       972
weighted avg       0.95      0.95      0.95       972



In [6]:
# --- ÉTAPE 7 : SAUVEGARDE (DEPLOYMENT COMPARATIF) ---

# On sauvegarde ce modèle juste pour la comparaison
filename_16 = 'modele_facile_16.pkl'

# On crée un petit paquet avec le modèle et les infos
package_16 = {
    'model': best_rf_16,
    'description': "Modèle entraîné sur le scenario facile (16% défauts)"
}

joblib.dump(package_16, filename_16)

print(f"✅ Modèle comparatif sauvegardé sous : {filename_16}")
print("👉 Utilisez ce fichier SEULEMENT pour montrer la différence de facilité.")
print("⚠️ Ne l'utilisez pas pour la production réelle (Data Leakage probable).")

✅ Modèle comparatif sauvegardé sous : modele_facile_16.pkl
👉 Utilisez ce fichier SEULEMENT pour montrer la différence de facilité.
⚠️ Ne l'utilisez pas pour la production réelle (Data Leakage probable).
